In [1]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings 
from pathlib import Path
import frontmatter

In [2]:
def load_markdown_folder(folder_path):
    documents = []

    for file_path in Path(folder_path).rglob("*.md"):   

        post = frontmatter.load(file_path)

        metadata = dict(post.metadata)

        content = post.content

        documents.append({
            "content": content,
            "metadata": {
                **metadata,
                "source": str(file_path),
                "file_name": file_path.name,
                "file_type": "markdown"
            }
        })

    return documents


documents = load_markdown_folder("./knowledge-base")

In [3]:
ddocs = []
for doc in documents:
    ddocs.append(
        Document(
            page_content=doc["content"],
            metadata=doc["metadata"]
        )
    )


In [4]:
ddocs

[Document(metadata={'document_id': 'RET-2026-01', 'title': 'Returns Policy', 'status': 'active', 'effective_date': datetime.date(2026, 4, 1), 'last_reviewed': datetime.date(2026, 7, 15), 'audience': 'customer', 'policy_authority': 'official', 'supersedes': 'RET-2024-01', 'source': 'knowledge-base\\01-returns-policy-current.md', 'file_name': '01-returns-policy-current.md', 'file_type': 'markdown'}, page_content='# Returns Policy\n\n## Standard return window\n\nCustomers on the standard plan may request a return within **30 calendar days of delivery**.\n\nTrailPlus members receive a different return window. See the TrailPlus Membership Policy. The membership must have been active when the order was placed.\n\n## Item condition\n\nA returned item must be unused, unwashed, and in resalable condition. Original tags, accessories, and packaging must be included when they were supplied with the item.\n\nTrying an item indoors for fit does not by itself make it ineligible. Visible wear, odors, 

In [5]:
split = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

In [6]:
split_docs = split.split_documents(ddocs)

VECTOR DB

In [7]:
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

In [9]:
embeddings = HuggingFaceEndpointEmbeddings(
    model="BAAI/bge-m3"
)

e:\Projects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
client = QdrantClient(path="qdrant.db")

In [11]:
collection_name = "knowlegedb"

In [12]:
client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
)


True

In [13]:
vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings,
)

In [14]:
vector_store.add_documents(split_docs)
# results = vector_store.similarity_search("pyhton", k=3)

['61434614af474ed19237b1b8933657ee',
 '54e2511f49524b5cbb2e41337c90462d',
 'dcdcb78b2d584316bcfa1132016905ff',
 '14e4425f4c8d477b8ba48a4f7c0665ba',
 'dd90fdd370864229996fe29d5aa585f3',
 'aaa18c7831ae47a8890bd14344bda0db',
 '8c34ab31f7234527b1b28fc3e2c22340',
 '18de758d96d54d359c806357fd538e0a',
 'd2d52d32eee446cd89799b3ede457f17',
 'e155295807374ebd89c6079f6889a2d2',
 '8fbc7a95770245399c6b61f7f073a714',
 '92fa058e51a84760918906b619bb2fe9',
 '9282cde5928b4c77800887e0b1f0a7b9',
 '5cf24592f8ef4ea0ab496205ffb250ac',
 '638f19b3fbf44af382ecb59a2e002088',
 'ebac39716bf4455ea7690f8b47678bda',
 '1fa0a4552f8a41ba860b6c3a954f6128',
 'e3bef73f43f1441ab429f71b91fdcf59',
 '81050392dd3b4f498e8582bd88891d1b',
 'f7a582d391444c0191dc80a429e5294b',
 '60b6cfa7c1a24af383838382160706bf']

In [15]:
vector_store

Query

In [16]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever, MultiQueryRetriever
from langchain_groq import ChatGroq

C:\Users\shega\AppData\Local\Temp\ipykernel_16608\1937584044.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


In [17]:
load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.0)


In [18]:
k=5

Hybrid Search + Multi Query Retriver

In [19]:
bm25_retriever = BM25Retriever.from_documents(split_docs,k=k) 
retriever = vector_store.as_retriever(search_kwargs={"k": k}) 

ensemble_retriever = EnsembleRetriever(retrievers=[retriever, bm25_retriever], weights=[0.5, 0.5])

multi_query_retriever = MultiQueryRetriever.from_llm(
        retriever=ensemble_retriever,
        llm=llm
    )

In [20]:
from langchain_core.messages import HumanMessage, SystemMessage,AIMessage

In [21]:
query = "My TrailPlus membership was active when I ordered. What is my return window?"

In [22]:
def HYDE(query: str) -> str:
    messages = [
        SystemMessage(content="""You are generating a hypothetical document to improve retrieval for a search system.
Given the following question, write a short passage that would plausibly contain the answer.
Write it as if it were an excerpt from a real document (e.g. a research paper, article, or technical report) — not as a direct answer to the question, and not addressed to the reader.
Do not hedge, do not say "I don't know," and do not include disclaimers.
Write confidently, using domain-appropriate terminology, even if some details are invented.
Keep it to 3-5 sentences."""),
        HumanMessage(content=query)
    ]
    response = llm.invoke(messages)
    return response.content
    

In [23]:
enhanced_query=HYDE(query)

In [24]:
enhanced_query

'TrailPlus members enjoy an extended return policy that differs from the standard consumer window. When a purchase is made while the membership is active, the customer is granted a 60‑day return period measured from the date of delivery. This extended window applies to all eligible merchandise, provided the items are returned in their original condition with proof of purchase. Non‑members, by contrast, retain the baseline 30‑day return timeframe.'

In [25]:

unique_docs = multi_query_retriever.invoke(enhanced_query)

In [26]:
seen = set()
unique_clean_docs = []
for doc in unique_docs:
    if doc.page_content not in seen:
        seen.add(doc.page_content)
        unique_clean_docs.append(doc.page_content)

Reranking

In [27]:
import cohere

In [28]:
load_dotenv()
reranking_model = cohere.ClientV2()
response = reranking_model.rerank(
        model="rerank-v3.5",
        query=query,
        documents=unique_clean_docs,
        top_n=7
    )

In [29]:
reranked_docs = []

for result in response.results:
        doc = unique_clean_docs[result.index]
        reranked_docs.append(doc)

In [30]:
reranked_docs

['# TrailPlus Membership Benefits\n\n## Return window\n\nA customer whose TrailPlus membership was active when an order was placed receives a **45-calendar-day return window from delivery** for eligible items.\n\nJoining TrailPlus after placing an order does not extend that order’s return window.\n\nFinal-sale restrictions, item-condition requirements, and warranty rules still apply.\n\n## Shipping benefit\n\nTrailPlus members receive free standard shipping on eligible United States orders without a minimum purchase amount.\n\nThe benefit does not cover expedited shipping, import duties, Canadian return postage, or other international charges.\n\n## Membership verification\n\nWhen membership status is not available to the agent, it should explain the standard policy and ask the customer to confirm whether TrailPlus was active on the order date. It must not assume membership based only on the customer requesting the benefit.',
 '# Returns Policy\n\n## Standard return window\n\nCustomers

In [31]:
response.results

[V2RerankResponseResultsItem(index=0, relevance_score=0.9466806),
 V2RerankResponseResultsItem(index=1, relevance_score=0.91288006),
 V2RerankResponseResultsItem(index=2, relevance_score=0.39431813),
 V2RerankResponseResultsItem(index=6, relevance_score=0.2566321),
 V2RerankResponseResultsItem(index=4, relevance_score=0.081381105),
 V2RerankResponseResultsItem(index=7, relevance_score=0.05886858),
 V2RerankResponseResultsItem(index=8, relevance_score=0.05771075)]

In [32]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""
Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {question}
""")

# Use reranked_docs directly — they are already strings (page_content)
context = "\n\n".join(reranked_docs)

chain = prompt | llm | StrOutputParser()


In [33]:
answer = chain.invoke({"context": context, "question": query})
print(answer)

Your return window is **45 calendar days from the date of delivery** for eligible items.
